# Explainable Multimodal Medical Diagnosis - V2 Upgrade

This notebook trains and evaluates the V2 Multimodal (EfficientNet-B0 + Bio_ClinicalBERT + Attention) model.

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/Medical_AI_Project/models', exist_ok=True)
print("✅ Google Drive connected!")

## 2. Clone Repository & Install Dependencies

In [ ]:
%%bash
cd /content
rm -rf Explainable-Multimodal-Medical-Diagnosis-Using-Chest-X-ray-Images-and-Clinical-Text-
git clone https://github.com/toqeer-ahmed/Explainable-Multimodal-Medical-Diagnosis-Using-Chest-X-ray-Images-and-Clinical-Text-.git project_code
cd project_code
pip install -r requirements.txt

## 3. Extract Dataset

In [ ]:
# Unzip your dataset from Google Drive
!unzip -q /content/drive/MyDrive/data.zip -d /content/project_code/

## 4. Run Preprocessing (Top 15 Classes)

In [ ]:
%%bash
cd /content/project_code
python src/data_preprocessing.py

## 5. Train V2 Model

In [ ]:
%%bash
cd /content/project_code
python src/train.py --epochs 10 --batch_size 16

## 6. Backup Model to Drive

In [ ]:
import shutil

source_model = '/content/project_code/outputs/models/best_model.pth'
dest_model = '/content/drive/MyDrive/Medical_AI_Project/models/best_model_V2.pth'

shutil.copy(source_model, dest_model)
print("🎉 SUCCESS! Model saved to Google Drive!")

## 7. Evaluate Model

In [ ]:
%%bash
cd /content/project_code
python src/evaluate.py

## 8. Visualize Explainability (Grad-CAM & BERT Attention)

In [ ]:
import sys
sys.path.append('/content/project_code')
import torch
import pandas as pd
import numpy as np
from src.fusion_model import MultimodalFusion
from src.data_loader import get_data_loaders
from src.explainability import generate_gradcam, get_text_attention, visualize_explainability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
df_labels = pd.read_csv('/content/project_code/data/processed/labels.csv')
num_classes = len(df_labels)

model = MultimodalFusion(num_classes=num_classes).to(device)
model.load_state_dict(torch.load('/content/project_code/outputs/models/best_model.pth'))
model.eval()

train_loader, val_loader, _ = get_data_loaders('/content/project_code/data/processed/dataset.csv', batch_size=1)
sample = next(iter(val_loader))
tokenizer = val_loader.dataset.tokenizer

images = sample['image'].to(device)
input_ids = sample['input_ids'].to(device)
attention_mask = sample['attention_mask'].to(device)

mean = np.array([0.485, 0.456, 0.406]).reshape((3, 1, 1))
std = np.array([0.229, 0.224, 0.225]).reshape((3, 1, 1))
img_np = images[0].cpu().numpy()
img_unnorm = (img_np * std) + mean
img_unnorm = np.transpose(img_unnorm, (1, 2, 0))
img_unnorm = np.clip(img_unnorm, 0, 1)

vis_image, target_idx = generate_gradcam(
    model=model, image_tensor=images, input_ids=input_ids, 
    attention_mask=attention_mask, original_image=img_unnorm, device=device
)

tokens, att_scores = get_text_attention(
    model=model, input_ids=input_ids, attention_mask=attention_mask,
    tokenizer=tokenizer, device=device
)

class_name = df_labels.iloc[target_idx]['class_name']
print(f"\nVisualizing Explainability for predicted disease: {class_name}")
visualize_explainability(vis_image, tokens, att_scores, class_name)